### **<h3 style="color:pink;"> RAG System — Week 2: Evaluation Framework**

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>

This notebook builds the evaluation foundation for our RAG system.

**What i'll do:**
- ✅ Verify all new packages (Groq, RAGAS, MLflow)
- ✅ Generate 200 QA pairs from our legal documents (ground truth)
- ✅ Run RAGAS evaluation pipeline
- ✅ Log baseline scores to MLflow

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup Verification**</span>

</div>

In [1]:
# ✅ Week 2 - Setup Verification
import mlflow
import groq
import ragas
import langchain_groq

print("✅ mlflow:", mlflow.__version__)
print("✅ groq:", groq.__version__)
print("✅ ragas:", ragas.__version__)
print("✅ langchain_groq: installed")
print("")
print("🎉 All Week 2 packages ready!")

✅ mlflow: 3.10.1
✅ groq: 0.37.1
✅ ragas: 0.4.3
✅ langchain_groq: installed

🎉 All Week 2 packages ready!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**API Configuration**</span>

</div>

In [ ]:
import os

# 🔑 Paste your Groq API key here
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"

# Test the connection
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Quick test
response = llm.invoke("Say hello in one word.")
print("✅ Groq connected!")
print(f"🤖 Model response: {response.content}")

✅ Groq connected!
🤖 Model response: Hello.


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Loading Our Documents**</span>

</div>

In [5]:
import json

# Load our chunks from Phase 1
with open("../data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks")
print(f"\n🔍 Sample chunk:")
print(f"   ID: {chunks[0]['chunk_id']}")
print(f"   Text: {chunks[0]['text'][:200]}...")


✅ Loaded 11954 chunks

🔍 Sample chunk:
   ID: legal_0000_chunk_000
   Text: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS. (a) Definitions.--In this section: (1) Business entity.--The term ``business entity'' means a firm, co...


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generating 200 QA Pairs (Ground Truth Dataset)**</span>

</div>

We'll use Groq (Llama 3) to automatically generate question-answer pairs from our legal chunks.
This becomes our **ground truth benchmark** — every future improvement will be measured against it.

In [6]:
import random
import time

random.seed(42)

# Pick 200 random chunks to generate QA pairs from
selected_chunks = random.sample(chunks, 200)

print(f"✅ Selected {len(selected_chunks)} chunks for QA generation")
print(f"📄 From {len(set(c['doc_id'] for c in selected_chunks))} different documents")
print(f"\n🔍 Sample selected chunk:")
print(selected_chunks[0]['text'][:300])

✅ Selected 200 chunks for QA generation
📄 From 158 different documents

🔍 Sample selected chunk:
. (d) Additional Terms.--The Secretary of Agriculture may require such additional terms or conditions in connection with the release of the reversionary interests under this section as the Secretary considers appropriate to protect the interests of the United States


Excellent! 200 chunks from 158 different documents — great diversity! 🎉

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Generating QA Pairs with Groq**</span>

</div>

⏳ This will take ~5-8 minutes — Groq is fast but we're making 200 API calls!

📝 What this code is doing:
We took 200 random chunks from our legal documents, and for each one we're asking Groq (Llama 3) to read it and generate:

- A question that can be answered from that chunk
- The correct answer to that question

For example, given this legal text:

"A business entity shall not be subject to civil liability if the use occurs outside the scope of business..."

Groq generates:
- Question: "Under what condition is a business entity not liable?"
- Answer: "When the use occurs outside the scope of business"

##### 📦 What we get at the end:

**A dataset of 200 question + answer + source chunk triplets that looks like this:**

- question: "What condition removes business liability?"

- answer:   "When use occurs outside business scope"

- context:  "A business entity shall not be subject..."

- doc_id:   "legal_0042"

In [7]:
import time
import json

def generate_qa_pair(chunk_text, llm):
    prompt = f"""You are a legal expert. Given this legal document excerpt, generate ONE question and its answer.

Document excerpt:
{chunk_text}

Rules:
- Question must be answerable ONLY from the excerpt above
- Answer must be concise (1-2 sentences)
- Focus on factual legal details

Respond in this exact JSON format:
{{"question": "your question here", "answer": "your answer here"}}

JSON only, no other text."""

    response = llm.invoke(prompt)
    return response.content

# Generate QA pairs with progress tracking
qa_pairs = []
failed = 0

print(f"⏳ Generating 200 QA pairs... (5-8 minutes)")
print(f"☕ Good time for a coffee break!\n")

start = time.time()

for i, chunk in enumerate(selected_chunks):
    try:
        raw = generate_qa_pair(chunk["text"], llm)
        
        # Clean and parse JSON
        raw = raw.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        
        parsed = json.loads(raw)
        
        qa_pairs.append({
            "question": parsed["question"],
            "answer": parsed["answer"],
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "context": chunk["text"]
        })
        
        # Progress update every 20
        if (i + 1) % 20 == 0:
            elapsed = time.time() - start
            print(f"   ✅ {i+1}/200 done ({elapsed:.0f}s elapsed)")
        
        time.sleep(0.1)  # avoid rate limiting
        
    except Exception as e:
        failed += 1
        if failed <= 3:
            print(f"   ⚠️ Failed chunk {i}: {e}")

elapsed = time.time() - start
print(f"\n✅ Generated {len(qa_pairs)} QA pairs in {elapsed:.0f} seconds")
print(f"⚠️  Failed: {failed}")

⏳ Generating 200 QA pairs... (5-8 minutes)
☕ Good time for a coffee break!

   ✅ 20/200 done (11s elapsed)
   ✅ 40/200 done (46s elapsed)
   ✅ 60/200 done (97s elapsed)
   ✅ 80/200 done (149s elapsed)
   ✅ 100/200 done (201s elapsed)
   ✅ 120/200 done (252s elapsed)
   ✅ 140/200 done (307s elapsed)
   ✅ 160/200 done (363s elapsed)
   ✅ 180/200 done (419s elapsed)
   ✅ 200/200 done (474s elapsed)

✅ Generated 200 QA pairs in 474 seconds
⚠️  Failed: 0


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Saving the Ground Truth Dataset**</span>

</div>

In [8]:
import os

# Save QA pairs to disk
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/qa_pairs.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(qa_pairs)} QA pairs to {output_path}")
print(f"📁 File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"\n🔍 Sample QA pair:")
print(f"   ❓ Q: {qa_pairs[0]['question']}")
print(f"   ✅ A: {qa_pairs[0]['answer']}")
print(f"   📄 Source: {qa_pairs[0]['doc_id']}")

✅ Saved 200 QA pairs to ../data/processed/qa_pairs.json
📁 File size: 140.8 KB

🔍 Sample QA pair:
   ❓ Q: Who is authorized to require additional terms or conditions in connection with the release of reversionary interests under this section?
   ✅ A: The Secretary of Agriculture is authorized to require such additional terms or conditions.
   📄 Source: legal_0441


Beautiful! 🎉 The QA pairs look great — real legal questions with precise answers!

Now let's build the RAGAS evaluation pipeline. This is the heart of Week 2!

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Building the RAGAS Evaluation Pipeline**</span>

</div>

RAGAS will score our RAG system on 3 key metrics:
- 📊 **Faithfulness** — Does the answer only use info from the retrieved context?
- 📊 **Answer Relevancy** — Is the answer relevant to the question?
- 📊 **Context Precision** — Are the retrieved chunks actually useful?

In [9]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load our FAISS index and embedding model from Phase 1
print("⏳ Loading FAISS index and embedding model...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("../data/embeddings/faiss_index.bin")

print(f"✅ FAISS index loaded! ({index.ntotal} vectors)")
print(f"✅ Embedding model loaded!")

# Define our search function
def search(query, top_k=3):
    query_vector = embedding_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_vector, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "chunk_id": chunks[idx]["chunk_id"],
            "distance": float(dist)
        })
    return results

print(f"\n🔍 Testing search...")
test = search("What are liability rules for business entities?")
print(f"✅ Search working! Got {len(test)} results")

⏳ Loading FAISS index and embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index loaded! (11954 vectors)
✅ Embedding model loaded!

🔍 Testing search...
✅ Search working! Got 3 results


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Running RAGAS Evaluation**</span>

</div>

⏳ We'll evaluate 50 QA pairs (representative sample) — running all 200 would take too long and cost more API calls.

In [10]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

# Wrap Groq LLM for RAGAS
ragas_llm = LangchainLLMWrapper(llm)

# Use 50 QA pairs for evaluation
eval_sample = qa_pairs[:50]

print("⏳ Preparing evaluation dataset...")

# Build the dataset RAGAS expects
eval_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for qa in eval_sample:
    # Retrieve context using our RAG search
    retrieved = search(qa["question"], top_k=3)
    contexts = [r["text"] for r in retrieved]
    
    eval_data["question"].append(qa["question"])
    eval_data["answer"].append(qa["answer"])
    eval_data["contexts"].append(contexts)
    eval_data["ground_truth"].append(qa["answer"])

dataset = Dataset.from_dict(eval_data)
print(f"✅ Dataset ready! {len(dataset)} samples")
print(f"\n🔍 Sample:")
print(f"   ❓ Q: {eval_data['question'][0]}")
print(f"   ✅ A: {eval_data['answer'][0]}")
print(f"   📄 Contexts retrieved: {len(eval_data['contexts'][0])}")

C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2709981174.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2709981174.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2709981174.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import c

⏳ Preparing evaluation dataset...
✅ Dataset ready! 50 samples

🔍 Sample:
   ❓ Q: Who is authorized to require additional terms or conditions in connection with the release of reversionary interests under this section?
   ✅ A: The Secretary of Agriculture is authorized to require such additional terms or conditions.
   📄 Contexts retrieved: 3


In [11]:
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

# Wrap Groq LLM for RAGAS
ragas_llm = LangchainLLMWrapper(llm)

# Use 50 QA pairs for evaluation
eval_sample = qa_pairs[:50]

print("⏳ Preparing evaluation dataset...")

# Build the dataset RAGAS expects
eval_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for qa in eval_sample:
    retrieved = search(qa["question"], top_k=3)
    contexts = [r["text"] for r in retrieved]
    
    eval_data["question"].append(qa["question"])
    eval_data["answer"].append(qa["answer"])
    eval_data["contexts"].append(contexts)
    eval_data["ground_truth"].append(qa["answer"])

dataset = Dataset.from_dict(eval_data)
print(f"✅ Dataset ready! {len(dataset)} samples")
print(f"\n🔍 Sample:")
print(f"   ❓ Q: {eval_data['question'][0]}")
print(f"   ✅ A: {eval_data['answer'][0]}")
print(f"   📄 Contexts retrieved: {len(eval_data['contexts'][0])}")

C:\Users\USER\AppData\Local\Temp\ipykernel_37756\3906591819.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


⏳ Preparing evaluation dataset...
✅ Dataset ready! 50 samples

🔍 Sample:
   ❓ Q: Who is authorized to require additional terms or conditions in connection with the release of reversionary interests under this section?
   ✅ A: The Secretary of Agriculture is authorized to require such additional terms or conditions.
   📄 Contexts retrieved: 3


In [13]:
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use LangchainLLMWrapper — most stable approach with Groq + RAGAS
ragas_llm = LangchainLLMWrapper(ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
))

# Use local embeddings (no API needed)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

print("✅ RAGAS LLM ready!")
print("✅ RAGAS Embeddings ready!")

C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2022298463.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatGroq(
C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2022298463.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAGAS LLM ready!
✅ RAGAS Embeddings ready!


C:\Users\USER\AppData\Local\Temp\ipykernel_37756\2022298463.py:16: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Running the Evaluation**</span>

</div>

⏳ This will take ~5-10 minutes — RAGAS is scoring each of our 50 QA pairs!

In simple terms, the code answers this question:

“How good is my RAG system at answering questions using retrieved documents?”

It does that by calculating evaluation metrics.

- Metric 1 — ***Faithfulness*** This checks: **Is the answer supported by the retrieved documents?**

- Metric 2 — ***Answer Relevancy*** This checks: **Does the answer actually respond to the question?**

- Metric 3 — ***Context Precision*** This checks: **Were the retrieved documents actually relevant?**

In [17]:
import warnings
warnings.filterwarnings("ignore")

# Use classic RAGAS metrics — fully compatible with Groq
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas import evaluate

# Assign our LLM and embeddings to the metrics
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
# Assign embeddings ( Embedding models convert text into vectors. ) ex: "What protects businesses?" → [0.23, -0.81, 0.12, ...]
answer_relevancy.embeddings = ragas_embeddings # Embeddings are used to compare semantic similarity between: question, answer
# Configure context precision (Again, the LLM is used to judge whether retrieved chunks are relevan, so that the LLM reads : Question, Retrieved chunk )
context_precision.llm = ragas_llm

print("✅ Metrics configured!")
print("⏳ Running RAGAS evaluation on 50 QA pairs...")
print("☕ Another good time for a coffee break!\n")

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
)

print("\n" + "="*50)
print("📊 BASELINE RAGAS SCORES:")
print("="*50)
print(f"   Faithfulness      : {results['faithfulness']:.4f}")
print(f"   Answer Relevancy  : {results['answer_relevancy']:.4f}")
print(f"   Context Precision : {results['context_precision']:.4f}")
print("="*50)

✅ Metrics configured!
⏳ Running RAGAS evaluation on 50 QA pairs...
☕ Another good time for a coffee break!



Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[18]: OutputParserException(Invalid json output: Here\u0027s the analysis of the complexity of each sentence in the answer:\n\nInput:\n{\n    \"question\": \"Who prescribes regulations for expenditures under this paragraph?\",\n    \"answer\": \"The Committee on House Administration of the House of Representatives.\"\n}\n\nOutput:\n{\n    \"statements\": [\n        \"The Committee on House Administration prescribes regulations.\",\n        \"The Committee on House Administration is of the House of Representatives.\"\n    ]\n}\n\nHere\u0027s the breakdown of each sentence into one or more fully understandable statements:\n\n1. \"The Committee on House Administration of the House of Representatives.\"\n   - This sentence ca


📊 BASELINE RAGAS SCORES:


TypeError: unsupported format string passed to list.__format__

The code above was the core step, RAGAS now loop through the dataset.

Pipeline for each example:

Question
↓
Retrieved chunks
↓
Model answer
↓
Judge LLM evaluates
↓
Metric score generated

- *Behind the scenes evaluation process*

- For each question: 
- - Question: 'What protects business from liability?'
- - Context retrieved: SECTION 1. LIABILITY OF BUSINESS ENTITIES...
- - Answer generated : Businesses are protected from civil liability when nonprofits use their facilities.

- Now theJudge LLM evaluates:
- - *Faitfulness* propmt example: Does the answer contain information not present in the context?
Score between 0 and 1.
- - The *LLM reason and returns something like* : Faithfulness = 0.92

- What the scores mean: 
Score range from: (0 → very bad
1 → perfect)

| Score | Meaning   |
| ----- | --------- |
| 0.9+  | excellent |
| 0.8   | good      |
| 0.6   | average   |
| <0.5  | poor      |

So these numbers tell you how well your RAG system performs.

In [18]:
import numpy as np

# Safely extract scores handling both single values and lists
def safe_score(val):
    if isinstance(val, list):
        # Filter out None values and average the rest
        clean = [v for v in val if v is not None]
        return np.mean(clean) if clean else 0.0
    return float(val) if val is not None else 0.0

faithfulness_score = safe_score(results['faithfulness'])
relevancy_score = safe_score(results['answer_relevancy'])
precision_score = safe_score(results['context_precision'])

print("=" * 50)
print("📊 BASELINE RAGAS SCORES:")
print("=" * 50)
print(f"   Faithfulness      : {faithfulness_score:.4f}")
print(f"   Answer Relevancy  : {relevancy_score:.4f}")
print(f"   Context Precision : {precision_score:.4f}")
print("=" * 50)
print(f"\n💡 Note: Some samples failed due to Groq rate limits")
print(f"   but we still got valid baseline scores!")

📊 BASELINE RAGAS SCORES:
   Faithfulness      : nan
   Answer Relevancy  : nan
   Context Precision : nan

💡 Note: Some samples failed due to Groq rate limits
   but we still got valid baseline scores!


In [19]:
# Check what's actually in results
print("Raw results:")
print(results)
print("\nResults type:", type(results))

# Check the dataframe
df = results.to_pandas()
print("\nDataframe columns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nScore summary:")
print(df.describe())

Raw results:
{'faithfulness': 0.5750, 'answer_relevancy': 0.6105, 'context_precision': 0.5784}

Results type: <class 'ragas.dataset_schema.EvaluationResult'>

Dataframe columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision']

First few rows:
                                          user_input  \
0  Who is authorized to require additional terms ...   
1  Who is responsible for paying the costs of app...   
2  What is the primary location of the lands desi...   
3          Who selects the chairperson of the Panel?   
4                What is the short title of the Act?   

                                  retrieved_contexts  \
0  [. (d) Additional Terms.--The Secretary of Agr...   
1  [. In order to save administrative costs to th...   
2  [attributes of the lands designated as wildern...   
3  [. (C) Two members appointed by the Minority L...   
4  [in connection with activities described in se...   

            

In [21]:
df = results.to_pandas()

faithfulness_score = df['faithfulness'].dropna().mean()
relevancy_score = df['answer_relevancy'].dropna().mean()
precision_score = df['context_precision'].dropna().mean()

print("=" * 50)
print("📊 BASELINE RAGAS SCORES:")
print("=" * 50)
print(f"   Faithfulness      : {faithfulness_score:.4f}")
print(f"   Answer Relevancy  : {relevancy_score:.4f}")
print(f"   Context Precision : {precision_score:.4f}")
print("=" * 50)
print(f"""
💡 Score Interpretation:
   0.0 - 0.4  → Poor
   0.4 - 0.6  → Baseline (expected for first run!)
   0.6 - 0.8  → Good
   0.8 - 1.0  → Excellent

🎯 Our system is already in the Baseline-to-Good range!
   We'll improve these scores week by week! 🚀
""")

📊 BASELINE RAGAS SCORES:
   Faithfulness      : 0.5750
   Answer Relevancy  : 0.6105
   Context Precision : 0.5784

💡 Score Interpretation:
   0.0 - 0.4  → Poor
   0.4 - 0.6  → Baseline (expected for first run!)
   0.6 - 0.8  → Good
   0.8 - 1.0  → Excellent

🎯 Our system is already in the Baseline-to-Good range!
   We'll improve these scores week by week! 🚀



<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Logging Baseline Scores to MLflow**</span>

</div>

MLflow tracks every experiment so we can see score progression over time.

In [22]:
import mlflow

# Start MLflow experiment
mlflow.set_experiment("RAG_Legal_Evaluation")

with mlflow.start_run(run_name="baseline_v1"):
    # Log parameters
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("chunk_size", 512)
    mlflow.log_param("chunk_overlap", 50)
    mlflow.log_param("top_k", 3)
    mlflow.log_param("eval_samples", 50)
    mlflow.log_param("llm_judge", "llama-3.1-8b-instant")

    # Log scores
    mlflow.log_metric("faithfulness", faithfulness_score)
    mlflow.log_metric("answer_relevancy", relevancy_score)
    mlflow.log_metric("context_precision", precision_score)

    print("✅ Baseline scores logged to MLflow!")
    print(f"\n📊 Run Summary:")
    print(f"   Experiment  : RAG_Legal_Evaluation")
    print(f"   Run name    : baseline_v1")
    print(f"   Faithfulness      : {faithfulness_score:.4f}")
    print(f"   Answer Relevancy  : {relevancy_score:.4f}")
    print(f"   Context Precision : {precision_score:.4f}")

2026/03/09 22:37:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/09 22:37:18 INFO mlflow.store.db.utils: Updating database tables
2026/03/09 22:37:19 INFO mlflow.tracking.fluent: Experiment with name 'RAG_Legal_Evaluation' does not exist. Creating a new experiment.


✅ Baseline scores logged to MLflow!

📊 Run Summary:
   Experiment  : RAG_Legal_Evaluation
   Run name    : baseline_v1
   Faithfulness      : 0.5750
   Answer Relevancy  : 0.6105
   Context Precision : 0.5784


PERFECT! 🎉 MLflow is running and your baseline scores are permanently logged!

Now let's launch the MLflow dashboard so you can see it visually

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Launching MLflow Dashboard**</span>

</div>

In [23]:
print("🚀 To view your MLflow dashboard:")
print("   1. Open a NEW PowerShell window")
print("   2. Navigate to your project:")
print("      cd C:\\Users\\USER\\Documents\\RAG_Project")
print("   3. Activate venv:")
print("      venv\\Scripts\\activate")
print("   4. Run:")
print("      mlflow ui")
print("   5. Open browser and go to:")
print("      http://127.0.0.1:5000")
print("")
print("✅ You'll see your RAG_Legal_Evaluation experiment")
print("✅ With baseline_v1 run and all 3 scores logged!")

🚀 To view your MLflow dashboard:
   1. Open a NEW PowerShell window
   2. Navigate to your project:
      cd C:\Users\USER\Documents\RAG_Project
   3. Activate venv:
      venv\Scripts\activate
   4. Run:
      mlflow ui
   5. Open browser and go to:
      http://127.0.0.1:5000

✅ You'll see your RAG_Legal_Evaluation experiment
✅ With baseline_v1 run and all 3 scores logged!


In [24]:
import mlflow

print("📁 MLflow tracking URI:", mlflow.get_tracking_uri())

📁 MLflow tracking URI: sqlite:///C:/Users/USER/Documents/RAG_Project/notebooks/mlflow.db


In [25]:
import mlflow

# Set correct tracking URI at project root
mlflow.set_tracking_uri("sqlite:///C:/Users/USER/Documents/RAG_Project/mlflow.db")

# Re-create experiment and log scores
mlflow.set_experiment("RAG_Legal_Evaluation")

with mlflow.start_run(run_name="baseline_v1"):
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("chunk_size", 512)
    mlflow.log_param("chunk_overlap", 50)
    mlflow.log_param("top_k", 3)
    mlflow.log_param("eval_samples", 50)
    mlflow.log_param("llm_judge", "llama-3.1-8b-instant")

    mlflow.log_metric("faithfulness", faithfulness_score)
    mlflow.log_metric("answer_relevancy", relevancy_score)
    mlflow.log_metric("context_precision", precision_score)

    print("✅ Scores re-logged to correct location!")
    print(f"📁 Location: C:/Users/USER/Documents/RAG_Project/mlflow.db")

2026/03/09 22:49:13 INFO mlflow.tracking.fluent: Experiment with name 'RAG_Legal_Evaluation' does not exist. Creating a new experiment.


✅ Scores re-logged to correct location!
📁 Location: C:/Users/USER/Documents/RAG_Project/mlflow.db


i opend a new powershell where i wrote the:

*cd C:\Users\USER\Documents\RAG_Project*

Then this:

*venv\Scripts\activate*

Then this:

*mlflow ui*

```
5. **Then I opened my  browser and go to:**
```
*http://127.0.0.1:5000*

i found on experiment : 

*RAG_Legal_Evaluation* and i clicked on it, then i clicked on *baseline_v1* and i found the metrics we calculated previously

| Metric | Value   | 
| ----- | --------- |
| Faithfulness  | 0.575 |
| answer_relevancy  | 0.6105427930347375      |
| context_precision   | 0.5784313725068627  |
____

|Parameter|Value|
|----|---|
|embedding_model|all-MiniLM-L6-v2|
|chunk_size|512|
|chunk_overlap|50|
|top_k|3|
|eval_samples|50|
|llm_judge|llama-3.1-8b-instant|

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Week 2 Summary**</span>

</div>

In [26]:
# Cell 30
print("=" * 60)
print("🎉 PHASE 1 - WEEK 2 COMPLETE!")
print("=" * 60)

print("""
📦 What we built today:
   ✅ Groq API connected (llama-3.1-8b-instant)
   ✅ 200 QA pairs generated from legal documents
   ✅ RAGAS evaluation pipeline built
   ✅ Baseline scores measured
   ✅ MLflow tracking dashboard running

📊 Baseline Scores (baseline_v1):
""")
print(f"   Faithfulness      : {faithfulness_score:.4f}")
print(f"   Answer Relevancy  : {relevancy_score:.4f}")
print(f"   Context Precision : {precision_score:.4f}")

print("""
📁 Files saved to disk:
   📄 qa_pairs.json   ← 200 ground truth QA pairs
   📄 mlflow.db       ← Experiment tracking database

🔜 Next Session — Week 3:
   → Advanced chunking strategies
   → Embedding model comparison
   → Beat these baseline scores!

============================================================
""")

🎉 PHASE 1 - WEEK 2 COMPLETE!

📦 What we built today:
   ✅ Groq API connected (llama-3.1-8b-instant)
   ✅ 200 QA pairs generated from legal documents
   ✅ RAGAS evaluation pipeline built
   ✅ Baseline scores measured
   ✅ MLflow tracking dashboard running

📊 Baseline Scores (baseline_v1):

   Faithfulness      : 0.5750
   Answer Relevancy  : 0.6105
   Context Precision : 0.5784

📁 Files saved to disk:
   📄 qa_pairs.json   ← 200 ground truth QA pairs
   📄 mlflow.db       ← Experiment tracking database

🔜 Next Session — Week 3:
   → Advanced chunking strategies
   → Embedding model comparison
   → Beat these baseline scores!




**How do we know if the system is giving good answers or bad answers?**

That's exactly what RAGAS does — it acts like a teacher grading our system:

Faithfulness 0.575 → "Is the answer based on the documents or is it making things up?" (57.5% faithful)

Answer Relevancy 0.610 → "Is the answer actually answering the question?" (61% relevant)

Context Precision 0.578 → "Did we retrieve the right document chunks?" (57.8% precise)

These scores are our report card. Every week we'll try to improve them!

🖥️ What is http://127.0.0.1:5000?

``127.0.0.1`` means your own computer. It's not a website on the internet — it's a mini website running locally on your machine.

When you ran mlflow ui in PowerShell, it started a small web server on your computer. Your browser just connected to it like any normal website, except it never leaves your computer.

🔗 How did RAG_Legal_Evaluation appear there?

Think of it like this:

|Jupyter Notebook      |    mlflow.db file     |     MLflow Dashboard |
|--------|--------|-------|
|(where you code)    |    (like a database)    |  (http://127.0.0.1:5000) |
|  You ran the evaluation code and logged scores       |    Scores got saved to this file on your computer      |       MLflow reads the file and shows it visually |


When you ran Cell 30 in Jupyter, it saved the scores into mlflow.db. When MLflow UI opened in your browser, it read that same file and displayed it as a nice dashboard!

🎊 WEEK 2 IS 100% COMPLETE!
Here's everything you accomplished today:

Built:

✅ 200 QA pairs (your permanent benchmark)

✅ RAGAS evaluation pipeline

✅ MLflow experiment tracking dashboard

✅ Baseline scores documented

Scores locked in:

📊 Faithfulness: 0.5750

📊 Answer Relevancy: 0.6105

📊 Context Precision: 0.5784

Saved to Git:

✅ All notebooks committed

✅ Repository clean

I run this on powershell ***``git show --stat 104df99``*** to be sure that i have saved everything on git
and i got : 

✅ ``notebooks/01_data_ingestion.ipynb`` — Week 1 notebook

✅ ``notebooks/02_evaluation.ipynb`` — Week 2 notebook

✅ ``data/processed/chunks.json`` — your 11,954 chunks

✅ ``data/processed/qa_pairs.json`` — your 200 QA pairs

✅ ``mlflow.db`` — your baseline scores

✅ ``.gitignore`` — your git configuration